# Speech to Text with Groq Whisper [Agent Patterns - Module 11]

> **MLCourse - Agentic AI - Agent Patterns**

A **voice agent** is not a new kind of agent. It is a text agent with two
extra stages bolted on:

```
audio in  ->  [ STT ]  ->  text  ->  [ your agent ]  ->  text  ->  [ TTS ]  ->  audio out
```

Everything you already know about prompts, tools and memory still applies to
the middle box. What is new is that the input is now *lossy* - the model
never sees what the user said, only what the transcriber **thought** they
said - and that changes how you design the agent.

This module runs entirely on **audio files**, not a live microphone, so it
works headless in a notebook and is fully reproducible.

### What you will learn

1. Generating a real `.wav` file with no microphone and no network.
2. Calling Groq's `whisper-large-v3` for transcription.
3. `verbose_json` - segments, timings, and what they are good for.
4. Measuring **Word Error Rate** against a known script.
5. The prompt parameter, and where transcription reliably fails.

### Key takeaways

- STT is a lossy front end. Design for a wrong transcript, not a perfect one.
- Numbers, IDs, names and spellings are where it breaks - and those are
  exactly what agents need.
- WER gives you a number instead of an impression. Measure it on *your*
  vocabulary.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import wave
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
CHAT_MODEL = "qwen/qwen3.8-27b"              # Groq-hosted; never OpenAI
STT_MODEL = "whisper-large-v3"               # Groq-hosted speech-to-text

HERE = Path.cwd().resolve()
AUDIO = HERE / "audio"
AUDIO.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Audio dir  : {AUDIO}")
print(f"Chat model : {CHAT_MODEL}")
print(f"STT model  : {STT_MODEL}")


### Making audio without a microphone


In [ ]:
# This module must run headless, so there is no mic. We SYNTHESISE the input
# audio with pyttsx3, which drives the operating system's built-in TTS voice
# (SAPI5 on Windows, NSSpeechSynthesizer on macOS, espeak on Linux).
#
# It is fully offline, needs no key, and gives us a real .wav file with real
# speech in it - which is exactly what the STT step needs.

import pyttsx3

def speak_to_file(text, path, rate=150):
    """Render `text` to a .wav file using the OS voice. Returns the Path."""
    path = Path(path)
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)          # words per minute
    engine.save_to_file(text, str(path))
    engine.runAndWait()
    engine.stop()
    return path

def wav_info(path):
    with wave.open(str(path)) as w:
        return {
            "seconds": round(w.getnframes() / w.getframerate(), 2),
            "sample_rate": w.getframerate(),
            "channels": w.getnchannels(),
            "bytes": Path(path).stat().st_size,
        }

print("speak_to_file() ready (offline OS voice)")


### 1. Making the input audio

There is no microphone in a headless notebook, so we synthesise the caller's
utterance with the operating system's own voice. This is deliberate, not a
shortcut:

- it is offline and keyless, so the notebook always runs;
- it produces a **real** waveform, so the STT step is doing real work;
- the script is known exactly, which lets us compute Word Error Rate in
  section 4 - impossible with a live recording.

The OS voice is also *harder* than a human in some ways (flat prosody,
clipped word boundaries), which makes it a decent stress test.

### Render a caller utterance to a .wav


In [ ]:
SCRIPT = ("Hi, this is Anita Sharma. My order number is four four seven one "
          "and it has not arrived yet. Can you check the delivery status "
          "and reschedule it for Friday morning?")

call_wav = speak_to_file(SCRIPT, AUDIO / "caller.wav")

print("wrote:", call_wav.name)
for k, v in wav_info(call_wav).items():
    print(f"  {k:12s} {v}")


### 2. Transcribing with Groq

Groq exposes an OpenAI-compatible transcription endpoint. The request is a
**multipart upload**: the audio goes in `files`, the parameters in `data`.
That trips people up - a JSON body silently gets you a 400.

`whisper-large-v3` is the accurate model. Groq also serves
`whisper-large-v3-turbo`, which is faster and slightly less accurate - a real
trade to make once you have measured WER on your own audio.

### Groq speech-to-text


In [ ]:
# whisper-large-v3 on Groq. Multipart upload: the file goes in `files`, the
# parameters go in `data`. Backoff included - the free tier is shared.

import requests

STT_URL = "https://api.groq.com/openai/v1/audio/transcriptions"

def transcribe(path, response_format="json", language="en"):
    """Send a .wav to Groq whisper-large-v3 and return the parsed response."""
    for attempt in range(5):
        with open(path, "rb") as fh:
            r = requests.post(
                STT_URL,
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                files={"file": (Path(path).name, fh, "audio/wav")},
                data={"model": STT_MODEL,
                      "response_format": response_format,
                      "language": language,
                      "temperature": 0},
                timeout=120,
            )
        if r.status_code == 200:
            return r.json()
        wait = 2 ** attempt + random.random()
        print(f"  HTTP {r.status_code}, retry {attempt+1} in {wait:.1f}s")
        time.sleep(wait)
    raise RuntimeError(f"STT failed: {r.status_code} {r.text[:300]}")

print("transcribe() ready")


### Transcribe


In [ ]:
t0 = time.time()
result = transcribe(call_wav)
stt_seconds = time.time() - t0

transcript = result["text"].strip()

print("transcript:")
print(" ", transcript)
print()
audio_seconds = wav_info(call_wav)["seconds"]
print(f"audio length : {audio_seconds:.2f}s")
print(f"STT latency  : {stt_seconds:.2f}s")
print(f"speed        : {audio_seconds / stt_seconds:.1f}x faster than realtime")


### 3. `verbose_json`: segments and timings

`response_format="verbose_json"` returns the transcript **plus** a list of
segments with start/end times and per-segment quality signals. You want this
for three things:

- **subtitling / alignment** - jumping to the audio for a given sentence;
- **diarisation-lite** - long pauses between segments often mark turn
  changes;
- **confidence** - `no_speech_prob` and `avg_logprob` flag segments the model
  was unsure about, which is your cue to ask the user to repeat.

### Segment-level output


In [ ]:
verbose = transcribe(call_wav, response_format="verbose_json")

print("top-level keys:", sorted(verbose)[:8])
print(f"duration: {verbose.get('duration')}s   language: {verbose.get('language')}")
print()

segs = verbose.get("segments", [])
print(f"{len(segs)} segment(s):\n")
for s in segs:
    print(f"  [{s['start']:>5.2f} - {s['end']:>5.2f}]  "
          f"no_speech={s.get('no_speech_prob', 0):.3f}  "
          f"avg_logprob={s.get('avg_logprob', 0):.2f}")
    print(f"      {s['text'].strip()}")


### Using the confidence signals

`avg_logprob` closer to 0 is more confident; strongly negative means the
model was guessing. `no_speech_prob` near 1 means that stretch was probably
silence or noise.

A practical rule for an agent: if a segment carrying a **critical value** (an
order number, an amount, a date) is low-confidence, do not act on it - read
it back and ask for confirmation. That pattern is built in notebook 03.

### 4. Measuring Word Error Rate

"The transcript looks fine" is not a measurement. WER is the standard number:
the edit distance between the reference and the hypothesis, in words, divided
by the reference length.

We know the script exactly, so we can compute it honestly.

### Word Error Rate


In [ ]:
# WER = (substitutions + insertions + deletions) / words in the reference.
# It is the standard ASR metric. 0.0 is perfect; 1.0 means every word wrong.
# Text is normalised first - case and punctuation are not transcription errors.

def normalise(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9' ]+", " ", text)
    return text.split()

def wer(reference, hypothesis):
    ref, hyp = normalise(reference), normalise(hypothesis)
    # classic Levenshtein over WORDS
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,        # deletion
                          d[i][j - 1] + 1,        # insertion
                          d[i - 1][j - 1] + cost) # substitution
    return d[len(ref)][len(hyp)] / max(1, len(ref))

# sanity-check the metric before trusting a number it produces
print(round(wer("the cat sat", "the cat sat"), 3), "expect 0.0")
print(round(wer("the cat sat", "the dog sat"), 3), "expect 0.333")


### Measure WER on our utterance


In [ ]:
score = wer(SCRIPT, transcript)

print("reference :", SCRIPT)
print()
print("hypothesis:", transcript)
print()
print(f"WER: {score:.1%}  ({len(normalise(SCRIPT))} reference words)")
print()
if score == 0:
    print("Perfect on this clip. Note that clean synthesised speech is the")
    print("easy case - real callers bring accents, noise and crosstalk.")
else:
    print("Non-zero. Look at WHICH words moved before deciding it matters:")
    print("'four four seven one' -> '4471' is a formatting difference, not a")
    print("comprehension failure, and a normaliser should absorb it.")


### Word-level diff


In [ ]:
# WER as a single number hides where the damage is. Always look at the diff.

import difflib

ref, hyp = normalise(SCRIPT), normalise(transcript)
for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=ref, b=hyp).get_opcodes():
    if tag == "equal":
        continue
    print(f"  {tag:9s} ref={' '.join(ref[i1:i2]) or '-':30s} hyp={' '.join(hyp[j1:j2]) or '-'}")
print("\n(no rows above means the transcript matched word for word)")


### 5. The `prompt` parameter

Whisper accepts an optional `prompt`: a short piece of text that biases
decoding. It is the cheapest accuracy win available, and it exists precisely
because of the failure mode that matters most to agents - **domain
vocabulary**.

Product names, drug names, ticket prefixes, your company's spelling of
things: none of it is in the model's prior. Feed a few in and they start
being recognised.

### Domain vocabulary, with and without a prompt


In [ ]:
JARGON = ("I need to reorder part N W two one hundred and the P T F E tape, "
          "and log it against ticket R M A dash eight eight one two.")

jargon_wav = speak_to_file(JARGON, AUDIO / "jargon.wav")

plain = transcribe(jargon_wav)["text"].strip()
print("without prompt:", plain)


### Same audio, with a vocabulary hint


In [ ]:
def transcribe_with_prompt(path, prompt):
    for attempt in range(5):
        with open(path, "rb") as fh:
            r = requests.post(
                STT_URL,
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                files={"file": (Path(path).name, fh, "audio/wav")},
                data={"model": STT_MODEL, "response_format": "json",
                      "language": "en", "temperature": 0, "prompt": prompt},
                timeout=120,
            )
        if r.status_code == 200:
            return r.json()["text"].strip()
        wait = 2 ** attempt + random.random()
        print(f"  HTTP {r.status_code}, retry {attempt+1} in {wait:.1f}s")
        time.sleep(wait)
    raise RuntimeError(f"STT failed: {r.status_code}")

HINT = ("Northwind Parts catalogue. SKUs look like NW-1001, NW-2100. "
        "Products include PTFE tape and compression valves. "
        "Return tickets look like RMA-8812.")

hinted = transcribe_with_prompt(jargon_wav, HINT)
print("with prompt   :", hinted)
print()

# Two references: what was SPOKEN, and what we actually want WRITTEN.
WRITTEN = ("I need to reorder part NW-2100 and the PTFE tape, "
           "and log it against ticket RMA-8812.")

print(f"{'':16s}{'vs spoken form':>16s}{'vs written form':>18s}")
print("-" * 50)
print(f"{'no prompt':16s}{wer(JARGON, plain):>15.1%}{wer(WRITTEN, plain):>18.1%}")
print(f"{'with prompt':16s}{wer(JARGON, hinted):>15.1%}{wer(WRITTEN, hinted):>18.1%}")
print("-" * 50)
print()
print("Read that table carefully - there are two lessons in it.")


### Two lessons from that table

**One: the prompt did not help here.** Whisper already resolved "N W two one
hundred" to `NW-2100` unaided. That is an honest result and worth reporting -
the prompt parameter is cheap insurance, not a guaranteed win, and on
vocabulary the model already knows it changes nothing. Measure before you
credit it. Where it *does* earn its place is genuinely unusual vocabulary:
invented product names, non-English surnames, internal codes with no
plausible English spelling.

**Two: your reference decides your number.** Against the spoken script the
WER looks terrible, because `NW-2100` is four written words where the caller
said nine. Against the written form we actually wanted, it is near zero. The
transcription did not change - the yardstick did.

So: **normalise both sides before scoring**, or score against the form your
downstream system consumes. A WER number without a stated normalisation
policy is not a measurement, it is a mood. This is the same discipline as the
metric design in module 09.

### Pitfalls recap

- **Sending JSON instead of multipart.** The transcription endpoint wants a
  file upload; a JSON body gets a 400.
- **Trusting the transcript.** It is a hypothesis. Numbers, IDs, names and
  spellings are the first casualties, and they are exactly what agents act on.
- **Judging quality by eye.** Measure WER on audio that looks like your real
  traffic, not on a clean studio clip.
- **Ignoring the confidence fields.** `verbose_json` tells you when the model
  was guessing. Use it to trigger a confirmation instead of a wrong action.
- **Unnormalised WER.** "four four seven one" vs "4471" is a formatting
  difference, not a transcription error. Decide the normalisation before you
  quote a number.
- **Forgetting `language`.** Leaving it unset lets Whisper auto-detect, and
  it will occasionally guess wrong on short or noisy clips.

### Next

Notebook 02 wraps a text agent around this and speaks the answer back.